# **SDG Text Classification**
## XLM-RoBERTa on SDGi Corpus

### Objective
Clean the text, tokenize it, and fine-tune a model to predict one or more of the 17 SDGs.

### Pipeline
1. Load and clean the text.
2. Prepare labels and separate training, validation and test data.
3. Tokenize the text using the pretrained model's tokenizer.
4. Fine-tune, evaluate and save the classifier.


Dataset: [UNDP SDGi Corpus](https://huggingface.co/datasets/UNDP/sdgi-corpus)

### 1. Environment Setup

#### Step 1. Install the required packages

In [3]:
%pip install "transformers==4.48.3" "datasets==3.2.0" "accelerate==1.3.0" "torch>=2.6,<3" "pyarrow>=14,<21" pandas sentencepiece scikit-learn

#### Step 1.1. Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### Step 2. Import libraries and set the model


In [5]:
from pathlib import Path
import re
import unicodedata
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments, set_seed
)
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score

set_seed(42)
MODEL_NAME = "FacebookAI/xlm-roberta-base"

### 2. Text Cleaning and Labels

#### Step 1. Load the dataset

In [6]:
train_df = pd.read_parquet("/content/drive/MyDrive/thesis/sdgi-corpus/data/train-00000-of-00001.parquet")
test_df = pd.read_parquet("/content/drive/MyDrive/thesis/sdgi-corpus/data/test-00000-of-00001.parquet")

print("Training texts:", len(train_df))
print("Test texts:", len(test_df))

Training texts: 5880
Test texts: 1470


#### Step 2. Clean the text

Keep punctuation, accents and complete words because the pretrained tokenizer expects natural text. thus we do not stem words or remove stopwords.

In [7]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [8]:
train_df["text"] = train_df["text"].fillna("").apply(clean_text)
test_df["text"] = test_df["text"].fillna("").apply(clean_text)
train_df = train_df[train_df["text"].ne("")].copy()
test_df = test_df[test_df["text"].ne("")].copy()

print(train_df["text"].iloc[0][:300])

END POVERTY IN ALL ITS FORMS EVERYWHERE 22 Poverty can be understood as a phenomenon that goes beyond monetary insu!ciency. A multidimensional conception of poverty takes into consideration minimum consumption of essential basic goods and services “as part of the set of opportunities that any societ


#### Step 3. Prepare the multi-label targets

The SDGi labels field lists goal numbers. Convert each list into 17 values in SDG 1-17 order: 1.0 for a recorded label and 0.0 otherwise. Floating-point targets are required for binary cross-entropy.

In [9]:
train_labels = [[float(i in labels) for i in range(1, 18)]
                for labels in train_df["labels"]]
test_labels = [[float(i in labels) for i in range(1, 18)]
               for labels in test_df["labels"]]

train_data = Dataset.from_dict({"text": train_df["text"].tolist(), "labels": train_labels})
test_data = Dataset.from_dict({"text": test_df["text"].tolist(), "labels": test_labels})

#### Step 4. Separate training, validation and test data

Save 10% of the official training split for validation. Keep the official test split untouched for final evaluation. This is a simple random split, not a source-report-grouped split.

In [10]:
split = train_data.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
validation_data = split["test"]

print(len(train_data), len(validation_data), len(test_data))

5292 588 1470


### 3. Tokenization



#### Step 1. Load the tokenizer

Use the tokenizer paired with the pretrained model. It converts text into input IDs and attention masks. Keep at most 512 tokens; longer texts are truncated in this baseline.

In [11]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_text(batch):
    return tokenizer(batch["text"], truncation=True, max_length=512)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

#### Step 2. Tokenize each split

Tokenize in batches and keep the labels.

In [12]:
train_data = train_data.map(tokenize_text, batched=True, remove_columns=["text"])
validation_data = validation_data.map(tokenize_text, batched=True, remove_columns=["text"])
test_data = test_data.map(tokenize_text, batched=True, remove_columns=["text"])

Map:   0%|          | 0/5292 [00:00<?, ? examples/s]

Map:   0%|          | 0/588 [00:00<?, ? examples/s]

Map:   0%|          | 0/1470 [00:00<?, ? examples/s]

In [ ]:
sample_texts = train_data["text"][:100]

word_counts = [len(text.split()) for text in sample_texts]
token_counts = [
    len(tokenizer(text, truncation=False)["input_ids"])
    for text in sample_texts
]

print("Average words:", sum(word_counts) / len(word_counts))
print("Average tokens:", sum(token_counts) / len(token_counts))
print("Texts exceeding 512 tokens:", sum(n > 512 for n in token_counts))

### 4. Model Fine-Tuning

#### Step 1. Load the classifier

Add 17 classification outputs. multi_label_classification uses binary cross-entropy so that each SDG can be predicted independently. The new classification head starts untrained.

In [ ]:
id2label = {i: f"SDG {i + 1}" for i in range(17)}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=17,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id={label: i for i, label in id2label.items()}
)

#### Step 2. Define classification metrics

Predict an SDG when its probability is at least 0.5. Calculate precision, recall and F1 for each text, then average over texts (average="samples"). Accuracy measures exact matches of the complete label set. Undefined precision or recall is set to zero. The threshold is a fixed baseline setting.

In [14]:
def compute_metrics(prediction):
    probabilities = torch.sigmoid(torch.tensor(prediction.predictions)).numpy()
    predicted_labels = (probabilities >= 0.5).astype(int)
    true_labels = prediction.label_ids

    return {
        "precision": precision_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "recall": recall_score(true_labels, predicted_labels, average="samples", zero_division=0),
        "accuracy": accuracy_score(true_labels, predicted_labels),
        "f1": f1_score(true_labels, predicted_labels, average="samples", zero_division=0)
    }

#### Step 3. Set training parameters

Start with a learning rate of 2^e-5, batch size of 4 and three epochs. Evaluate after each epoch and keep the checkpoint with the lowest validation loss. These values are starting settings, not tuned results.

In [21]:
from transformers import EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="xlm_roberta_sdgi/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=20,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=1,
    report_to="none",
    dataloader_pin_memory=False,
    seed=42
)

#### Step 4. Fine-tune the model

The Trainer handles batching, optimization and validation. `DataCollatorWithPadding` pads texts to the longest sequence in each batch.

In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=validation_data,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,Accuracy,F1
1,0.072500,0.092528,0.941213,0.901774,0.862245,0.908163
2,0.063200,0.078515,0.934825,0.935940,0.855442,0.925647
3,0.051300,0.072592,0.954287,0.932653,0.875850,0.933819
4,0.039200,0.066773,0.947317,0.931884,0.869048,0.931520
5,0.030300,0.074189,0.949938,0.927402,0.877551,0.928720
6,0.027200,0.068896,0.952725,0.936022,0.880952,0.936659


### 5. Evaluation and Saving

#### Step 1. Evaluate on the test set

Evaluate once after fine-tuning.

In [ ]:
test_results = trainer.evaluate(test_data, metric_key_prefix="test")

In [ ]:
test_results

#### Step 2. Save the classifier

Save the model and tokenizer together so that the same text processing can be used when making predictions.

In [ ]:
trainer.save_model("/content/drive/MyDrive/thesis/xlm_roberta_sdgi/model")
tokenizer.save_pretrained("/content/drive/MyDrive/thesis/xlm_roberta_sdgi/model")